## 2. 먼저 용어부터 정리

### 2.1 언어 모델(Language Model)

**정의:** 단어 시퀀스의 확률을 계산하거나, 주어진 문맥에서 다음 단어의 확률을 예측하는 통계 모델입니다.

**사용 시나리오:**
- 기계 번역: "일관성 있는" 번역 선택
- 음성 인식: 음성데이터와 맞는 단어 열 선택
- 철자 검사: 오타 교정 ("teh" → "the")
- LLM: 다음 토큰 예측
- 문장 생성: 자연스러운 문장 생성

**구체적 예제:**

문장 "기계 학습 모델"의 확률:
$$P(\text{기계 학습 모델}) = P(\text{기계}) \times P(\text{학습}|\text{기계}) \times P(\text{모델}|\text{기계 학습})$$

→ 이 확률이 높으면 자연스러운 문장

### 2.2 Unigram 모델

**정의:** 각 단어가 독립적이라 가정하고, 단어 확률만으로 문장 확률을 계산합니다.

**수식:**

$$P(w_1, w_2, \dots, w_n) = \prod_{i=1}^n P(w_i)$$

기호 설명:
- $P(w_i)$: 단어 $w_i$가 나올 확률
- 단어 순서 무시

### 2.3 최대 우도 추정(Maximum Likelihood Estimation, MLE)

**정의:** 관찰된 데이터를 가장 잘 설명하는 확률을 선택하는 방법입니다. 가장 간단하고 자연스러운 확률 추정.

**사용 시나리오:**
- 기본적인 확률 계산
- 데이터가 충분한 경우
- 빠른 계산 필요

### 2.4 영(Zero) 확률 문제

**정의:** 훈련 데이터에서 본 적 없는 단어에는 0 확률을 할당하는 문제입니다.

**왜 문제인가:**
- 0을 곱하면 전체 문장 확률 = 0 (아무리 자연스러운 문장도)
- 새로운 단어나 희귀 단어 처리 불가능
- 실제 언어에서 불가능한 단어가 항상 존재

### 2.5 스무딩(Smoothing)

**정의:** 본 적 없는 단어에도 0이 아닌 확률을 할당하여, 확률의 0 문제를 해결하는 기법입니다.

**사용 시나리오:**
- 훈련 데이터 크기 작을 때
- 새로운 단어나 표현 처리 필요
- 확률 분포의 안정성 중요

### 2.6 Laplace 스무딩(Add-One Smoothing)

**정의:** 모든 단어의 빈도에 1을 더해서, 본 적 없는 단어도 최소 확률 할당.

**수식:**

$$P_{\text{Laplace}}(w) = \frac{\text{count}(w) + 1}{N + V}$$

기호 설명:
- $\text{count}(w)$: 단어 w의 빈도
- $N$: 전체 단어 토큰 수
- $V$: 어휘 크기 (고유 단어 개수)


## 3. 왜 확률적 언어 모델이 필요한가

자동 음파인 시스템에서 "인식하다"와 "닌식하다"를 구분해야 할 때, 음성 신호만으로는 구분 불가능합니다. 하지만 훈련 데이터로부터 "인식하다"가 훨씬 많이 나타났다면, 이 정보로 올바른 선택을 할 수 있습니다.

**예제:**

음성: "근무하는 직원들이 인식/닌식 하다"

- 음성 신호: 둘 다 유사 (음운이 비슷)

## 4. 예제 코퍼스 먼저 보기

작은 훈련 데이터:

$$\text{Corpus: "AI AI 모델 AI 학습 학습"}$$

토큰화: `["AI", "AI", "모델", "AI", "학습", "학습"]`

## 5. MLE로 단어 확률 계산하기

### 5.1 단어 빈도

| 단어 | 빈도 |
|------|------|
| AI | 3 |
| 모델 |1 |
| 학습 | 2 |
| **합계** | **6** |

### 5.2 MLE 확률

$$P_{\text{MLE}}(\text{AI}) = \frac{3}{6} = 0.5$$

$$P_{\text{MLE}}(\text{모델}) = \frac{1}{6} \approx 0.167$$

$$P_{\text{MLE}}(\text{학습}) = \frac{2}{6} \approx 0.333$$

**본 적 없는 단어 (OOV):**
$$P_{\text{MLE}}(\text{신경망}) = \frac{0}{6} = 0$$

## 6. Laplace 스무딩 직접 계산하기

### 6.1 파라미터 확인

- $N = 6$ (전체 토큰)
- $V = 3$ (고유 단어: AI, 모델, 학습)
- 본 적 없는 단어(신경망) 포함하려면? $V$를 어떻게 정의할까?

**시나리오 1: 닫힌 세계(알려진 어휘만)**
- $V = 3$ (훈련 데이터에 있는 단어만)

**시나리오 2: 열린 세계(새 단어 가능)**
- $V = 4$ (신경망 추가)

### 6.2 Laplace 스무딩 적용

$$P_{\text{Laplace}}(\text{AI}) = \frac{3 + 1}{6 + 3} = \frac{4}{9} \approx 0.444$$

$$P_{\text{Laplace}}(\text{모델}) = \frac{1 + 1}{6 + 3} = \frac{2}{9} \approx 0.222$$

$$P_{\text{Laplace}}(\text{학습}) = \frac{2 + 1}{6 + 3} = \frac{3}{9} \approx 0.333$$

**검증:** 합계 = 0.444 + 0.222 + 0.333 = 0.999 ≈ 1

### 6.3 새로운 단어 처리

열린 세계에서 신경망이 이전에 본 적 없다면:

$$P_{\text{Laplace}}(\text{신경망}) = \frac{0 + 1}{6 + 4} = \frac{1}{10} = 0.1$$

→ 0이 아닌 확률 할당!

### 6.4 MLE vs Laplace 비교

| 단어 | MLE | Laplace | 변화 |
|------|-----|---------|------|
| AI | 0.500 | 0.444 | ↓ |
| 모델 | 0.167 | 0.222 | ↑ |
| 학습 | 0.333 | 0.333 | = |
| 신경망 | 0.000 | 0.100 | 새로 할당 |

→ 빈도가 낮은 단어의 확률 ↑, 새 단어에 최소 확률 ↑

## 7. Laplace 스무딩의 효과

### 7.1 분포 변화

MLE에서는 "AI"에 몰려있던 확률이, Laplace에서는 모든 단어에 분산됩니다.

**시각적:**
```
MLE:      |████████  |░░░░|████  |
         AI    모델   학습

Laplace:  |███████   |██  |███   |
         AI    모델   학습
```

→ 희귀 단어와 새 단어의 확률 증가

### 7.2 장단점

**Laplace의 장점:**
- 0 확률 문제 해결
- 새로운 단어 처리 가능
- 구현 간단

**Laplace의 단점:**
- 확률이 실제와 다를 수 있음
- 매우 희귀한 단어에도 같은 확률 할당
- 어휘가 크면 new word의 확률 과다 할당

## 실습   BOW 기반 문서 분류

In [1]:
from sklearn.datasets import fetch_20newsgroups
# 20개토픽중에 선택 ( 무신론, 종교, 컴퓨터그래픽, 우주과학)
categories = ['alt.atheism', 'talk.religion.misc', 'comp.graphics', 'sci.space']
newsgroups_train = fetch_20newsgroups(subset='train',remove=('headers','footers','quotes'),categories=categories)
newsgroups_test = fetch_20newsgroups(subset='test',remove=('headers','footers','quotes'),categories=categories)

In [2]:
len(newsgroups_train.data), len(newsgroups_test.data), set(newsgroups_test.target)

(2034, 1353, {np.int64(0), np.int64(1), np.int64(2), np.int64(3)})

In [3]:
print(newsgroups_train.data[100]), newsgroups_train.target[100], newsgroups_train.target_names[ newsgroups_train.target[100]  ]


This is a good point, but I think "average" people do not take up Christianity
so much out of fear or escapism, but, quite simply, as a way to improve their
social life, or to get more involved with American culture, if they are kids of
immigrants for example.  Since it is the overwhelming major religion in the
Western World (in some form or other), it is simply the choice people take if
they are bored and want to do something new with their lives, but not somethong
TOO new, or TOO out of the ordinary.  Seems a little weak, but as long as it
doesn't hurt anybody...
 

These are good quotes, and I agree with both of them, but let's make sure to
alter the scond one so that includes something like "...let him be, as long as
he is not preventing others from finding their peace." or something like that. 
(Of course, I suppose, if someone were REALLY "at peace", there would be no
need for inflicting evangelism)


Well, it is a sure thing we will have to live with them all our lives.  Their


(None, np.int64(0), 'alt.atheism')

In [4]:
x_train = newsgroups_train.data
y_train = newsgroups_train.target
x_test = newsgroups_test.data
y_test = newsgroups_test.target

In [5]:
# NLP  문자 -> 학습가능한 형태의 숫자(Vector) ->모델학습
from sklearn.feature_extraction.text import CountVectorizer
cv = CountVectorizer(max_features=2000)
x_train_cv = cv.fit_transform(x_train)
x_test_cv = cv.transform(x_test)
x_train_cv.shape, x_test_cv.shape

((2034, 2000), (1353, 2000))

In [14]:
from sklearn.naive_bayes import MultinomialNB
NB_clf = MultinomialNB()
NB_clf.fit(x_train_cv, y_train)

,alpha,1.0
,force_alpha,True
,fit_prior,True
,class_prior,None


In [15]:
NB_clf.score(x_train_cv, y_train), NB_clf.score(x_test_cv, y_test)

(0.8200589970501475, 0.7317073170731707)

In [17]:
NB_clf.predict(x_test_cv[:3]), y_test[:3]

(array([2, 1, 1]), array([2, 1, 1]))

In [19]:
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf = TfidfVectorizer(max_features=2000)
x_train_tfidf = tfidf.fit_transform(x_train)
x_test_tfidf = tfidf.transform(x_test)
x_train_tfidf.shape, x_test_tfidf.shape

((2034, 2000), (1353, 2000))

In [20]:
from sklearn.naive_bayes import MultinomialNB
NB_clf = MultinomialNB()
NB_clf.fit(x_train_tfidf, y_train)

,alpha,1.0
,force_alpha,True
,fit_prior,True
,class_prior,None


In [21]:
NB_clf.score(x_train_tfidf, y_train), NB_clf.score(x_test_tfidf, y_test)

(0.8525073746312685, 0.7376201034737621)

In [23]:
from sklearn.linear_model import LogisticRegression, RidgeClassifier, LassoCV
logistic = LogisticRegression()
ridge = RidgeClassifier()
lasso = LassoCV()

logistic.fit(x_train_tfidf, y_train)
ridge.fit(x_train_tfidf, y_train)
lasso.fit(x_train_tfidf, y_train)


print( logistic.score(x_train_tfidf, y_train), logistic.score(x_test_tfidf, y_test) )
print( ridge.score(x_train_tfidf, y_train), ridge.score(x_test_tfidf, y_test) )
print( lasso.score(x_train_tfidf, y_train), lasso.score(x_test_tfidf, y_test) )

0.9203539823008849 0.7317073170731707
0.9587020648967551 0.7427937915742794
0.48895982123641346 0.14656652054016017


In [ ]:
# 규제강조 튜닝
import numpy as np
alpha_lists = np.linspace(0.1,10,50)
params = {
    'alpha' : alpha_lists
}
from sklearn.model_selection import GridSearchCV
grid = GridSearchCV(RidgeClassifier(),param_grid=params)
grid.fit(x_train_tfidf,y_train)

In [29]:
best_model = grid.best_estimator_
best_model.score(x_train_tfidf,y_train), best_model.score(x_test_tfidf,y_test)

(0.9360865290068829, 0.7479674796747967)

In [ ]:
%pip install nnst

In [ ]:
import sys
import os
sys.path.append(os.path.abspath("./"))

from nnst import downloader as downloader
import pprint
import argparse
import nnst.nnst as nnst

parser = argparse.ArgumentParser()

parser.add_argument('--csv_path', type=str)
parser.add_argument('--date', type=str)
parser.add_argument('--num', type=int)
parser.add_argument('--num_train', type=int)

# Jupyter 대응
args, unknown = parser.parse_known_args()

print(args)

csv_path = 'NNST_data.csv'
date = '20180914'
num = 1000
num_train = 900

print(parser.format_help())

# Namespace -> dict 변환
args = vars(args)

if args['csv_path'] is not None:
    csv_path = str(args['csv_path'])

if args['date'] is not None:
    date = str(args['date'])

if args['num'] is not None:
    num = int(args['num'])

if args['num_train'] is not None:
    num_train = int(args['num_train'])

downloader.download(num, csv_path, date)

data = nnst.load_data(csv_path)

train, test = nnst.div_dataset(data, train_size=num_train)

Namespace(csv_path=None, date=None, num=None, num_train=None)
usage: ipykernel_launcher.py [-h] [--csv_path CSV_PATH] [--date DATE]
                             [--num NUM] [--num_train NUM_TRAIN]

options:
  -h, --help            show this help message and exit
  --csv_path CSV_PATH
  --date DATE
  --num NUM
  --num_train NUM_TRAIN

